|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 0:</h2>|<h1>From a Program to a Model<h1>|
|<h2>Section:</h2>|<h1>The model<h1>|
|<h2>Lecture:</h2>|<h1><b>Code challenge: write the loop<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# Find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

# The notebook runs on a GPU if there is one, and on the CPU if not.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' else torch.float32
MODEL_NAME = 'Qwen/Qwen3-0.6B'

Write the generation loop yourself, from the model call up.

You need the model and the tokenizer, nothing more. The checks at the end of
each exercise compare your code with the library.

In [ ]:
### run this cell
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE).to(DEVICE).eval()
print(f'{MODEL_NAME} on {DEVICE}')

# Exercise 1: the logits of the next token

Call the model on a list of token ids. Return the logits of the LAST position
only: one score for each token of the vocabulary.

In [ ]:
def next_token_logits(token_ids):
  """token_ids: a list of ints. -> a float tensor of shape (vocabulary,)."""
  

token_ids = tokenizer('The capital of France is').input_ids
logits = next_token_logits(token_ids)
assert logits.shape == (model.config.vocab_size,)
print('the most likely next token:', repr(tokenizer.decode([int(logits.argmax())])))

# Exercise 2: the greedy loop

Append the token with the highest logit, and call the model again. Stop at
EOS, or after `max_new_tokens` tokens. Return only the NEW token ids.

In [ ]:
def generate_greedy(prompt, max_new_tokens):
  """-> the list of new token ids, without the prompt."""
  token_ids = tokenizer(prompt).input_ids
  new_ids = []
  
  return new_ids

prompt = 'A list of three prime numbers:'
new_ids = generate_greedy(prompt, max_new_tokens=16)
prompt_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(DEVICE)
with torch.no_grad():
  expected = model.generate(prompt_ids, max_new_tokens=16, do_sample=False,
                            use_cache=False)[0, prompt_ids.shape[1]:]
print(repr(tokenizer.decode(new_ids)))
print('the same as generate():', new_ids == expected.tolist())

# Exercise 3: sampling

Write a sampler with a temperature. Temperature 0 must mean greedy. Then prove
that it is correct: draw many samples from a small set of logits, and compare
the counts with softmax.

In [ ]:
def sample_next(logits, temperature, generator):
  """-> one token id, chosen at random with softmax(logits / temperature)."""
  

NUM_DRAWS = 20_000
logits = torch.tensor([2.0, 1.0, 0.5, 0.0, -1.0], device=DEVICE)
generator = torch.Generator(device=DEVICE).manual_seed(0)
counts = torch.zeros(len(logits))
# Draw NUM_DRAWS samples at temperature 1, and count each token.

expected = torch.softmax(logits, -1).cpu()
sampled = counts / NUM_DRAWS
print(f"{'token':>6} {'expected':>9} {'sampled':>9}")
for token, (wanted, got) in enumerate(zip(expected.tolist(), sampled.tolist())):
  print(f'{token:>6} {wanted:>9.3f} {got:>9.3f}')
print(f'\nmax error {(sampled - expected).abs().max():.4f}')
print('temperature 0 is greedy:', sample_next(logits, 0, generator) == int(logits.argmax()))

### Before you open the solution

1. Your loop in Exercise 2 calls the model with the FULL list each time. How
   many tokens does it process to make 16 new tokens after an 8-token prompt?
2. Exercise 3 needs 20,000 draws for an error of about 1%. How many draws do
   you need for 0.1%? Why?
3. A temperature of 0.01 is almost greedy. What goes wrong numerically if you
   divide the logits by 0.00001?